# Nowcast + Scenario Workflow

This notebook demonstrates a **nowcasting-to-scenario pipeline**: predict the present,
then condition the future on alternative scenarios.

The workflow bridges two temporal domains:
1. **Nowcasting** — estimate the current quarter using high-frequency indicators
2. **Forecasting + Scenarios** — project forward from the nowcast under policy assumptions

**Pipeline**: Nowcast -> News Decomposition -> Bridge to Forecast -> Scenario Analysis -> Probability Assessment -> Monitoring

**Datasets used**: `mixed_freq.csv` (Phase 6), `us_macro_quarterly.csv` (Phase 5)

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# forecastbox nowcasting
from forecastbox.nowcasting import DFMNowcaster, NewsDecomposition

# forecastbox scenarios
from forecastbox.scenarios import (
    SimpleVAR,
    ConditionalForecast,
    ScenarioBuilder,
    MonteCarlo,
    FanChart,
)

# forecastbox metrics
from forecastbox.metrics import mae, rmse

# Helpers
sys.path.insert(0, "..")
from utils.helpers import load_all_datasets

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("All modules loaded successfully.")

## 1. The Nowcasting-Forecasting Bridge

In macroeconomic practice, the **present** is unknown: GDP is released with a 1-3 month
delay. Nowcasting fills this gap using timely monthly indicators.

Once we have a nowcast of the current quarter, we can **bridge** to a longer-horizon
forecast and attach **scenarios** for policy analysis:

```
Monthly data  -->  DFM Nowcast  -->  Bridge to VAR  -->  Scenarios  -->  Probabilities
(ragged edge)     (current GDP)    (forecast GDP)    (hawkish/dovish)   (P(recession))
```

This is exactly the pipeline central banks use for real-time assessment.

In [ ]:
# Load datasets
datasets = load_all_datasets()
mixed_freq = datasets["mixed_freq"]
us_macro = datasets["us_macro_quarterly"]

print("=== Mixed-Frequency Dataset ===")
print(f"Shape: {mixed_freq.shape}")
print(f"Date range: {mixed_freq.index[0]} to {mixed_freq.index[-1]}")
print(f"Columns: {list(mixed_freq.columns)}")
print(f"\nMissing values per column:")
print(mixed_freq.isna().sum())

print("\n=== US Macro Quarterly ===")
print(f"Shape: {us_macro.shape}")
print(f"Columns: {list(us_macro.columns)}")

# Visualize the ragged edge
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, mixed_freq.columns):
    series = mixed_freq[col].dropna()
    ax.plot(series.index, series.values, "steelblue", linewidth=1.2)
    ax.set_title(col.replace("_", " ").title(), fontsize=11)
    ax.grid(True, alpha=0.3)
    # Mark missing periods
    nan_mask = mixed_freq[col].isna()
    if nan_mask.any():
        for idx in mixed_freq.index[nan_mask]:
            ax.axvline(idx, color="red", alpha=0.1, linewidth=0.5)
fig.suptitle("Mixed-Frequency Data with Ragged Edge", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nRagged Edge (last 6 rows):")
print(mixed_freq.tail(6).to_string())

## 2. Step 1: Nowcast Current Quarter

We use a **Dynamic Factor Model (DFM)** to nowcast current-quarter GDP growth.
The DFM extracts latent factors from the mixed-frequency panel and handles the
ragged edge via the Kalman filter.

The Mariano-Murasawa (2003) accumulator links the quarterly GDP to the monthly factor.

In [ ]:
# Define frequency map
frequency_map = {
    "industrial_production": "M",
    "retail_sales": "M",
    "confidence_index": "M",
    "gdp_growth": "Q",
}

# Fit DFM
dfm = DFMNowcaster(
    n_factors=1,
    factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum",
    em_iterations=100,
    em_tol=1e-6,
)
dfm.fit(mixed_freq)
print(dfm)

# Nowcast
nowcast = dfm.nowcast(target="gdp_growth")
print(f"\n=== GDP Growth Nowcast ===")
print(f"Point estimate: {nowcast.point[0]:.4f}")
print(f"80% CI: [{nowcast.lower_80[0]:.4f}, {nowcast.upper_80[0]:.4f}]")
print(f"95% CI: [{nowcast.lower_95[0]:.4f}, {nowcast.upper_95[0]:.4f}]")

# Factor loadings
print("\nFactor Loadings:")
print(dfm.loadings())

# Plot factor vs GDP
factors = dfm.factors()
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(factors.index, factors["factor_1"], "steelblue", linewidth=2, label="Latent Factor")
ax1.set_ylabel("Factor", color="steelblue")
ax2 = ax1.twinx()
gdp_obs = mixed_freq["gdp_growth"].dropna()
ax2.scatter(gdp_obs.index, gdp_obs.values, color="darkorange", s=40, zorder=5, label="GDP (quarterly)")
# Mark nowcast
ax2.scatter([mixed_freq.index[-1]], [nowcast.point[0]], color="red", s=100, zorder=6,
            marker="*", label=f"Nowcast: {nowcast.point[0]:.2f}")
ax2.set_ylabel("GDP Growth", color="darkorange")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
ax1.set_title("DFM: Latent Factor vs GDP Growth (with Nowcast)", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Step 2: News Decomposition

When new data arrives, the nowcast revises. The **news decomposition** (Banbura & Modugno 2014)
tells us exactly **which data releases** drove the revision and by how much.

$$\Delta \hat{y} = \sum_i w_i \cdot (x_i^{\text{new}} - E[x_i | \Omega_{\text{old}}])$$

This is critical for central bank communication: "GDP was revised up because industrial
production surprised positively."

In [ ]:
# Simulate old vs new information sets
# Old: more missing data (before latest releases)
old_data = mixed_freq.copy()
old_data.iloc[-3:, old_data.columns.get_loc("industrial_production")] = np.nan
old_data.iloc[-3:, old_data.columns.get_loc("retail_sales")] = np.nan
old_data.iloc[-2:, old_data.columns.get_loc("confidence_index")] = np.nan

# New: some indicators updated (but not GDP)
new_data = mixed_freq.copy()
new_data.iloc[-1:, new_data.columns.get_loc("industrial_production")] = np.nan
new_data.iloc[-2:, new_data.columns.get_loc("retail_sales")] = np.nan

# Perform news decomposition
news_decomp = NewsDecomposition(dfm)
news_result = news_decomp.decompose(old_data, new_data, target="gdp_growth")

print("=== News Decomposition ===")
print(news_result.summary())

# Visualize contributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

news_result.plot_contributions(ax=axes[0])
axes[0].set_title("Contributions to Nowcast Revision", fontsize=12, fontweight="bold")

news_result.plot_waterfall(ax=axes[1])
axes[1].set_title("Waterfall: Old Nowcast -> New Nowcast", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"\nOld nowcast: {news_result.old_nowcast:.4f}")
print(f"New nowcast: {news_result.new_nowcast:.4f}")
print(f"Revision:    {news_result.total_revision:+.4f}")

## 4. Step 3: Bridge to Forecast

The nowcast gives us the **current quarter**. To forecast the future, we use a **VAR model**
on quarterly data, conditioning the first period on the nowcast.

This bridges the DFM's mixed-frequency nowcast into the VAR's multivariate forecast.

In [ ]:
# Build VAR on quarterly US macro data
var_vars = ["gdp_growth", "inflation", "fed_funds", "unemployment"]
endog = us_macro[var_vars].dropna().values
var_model = SimpleVAR(endog, p_order=2, var_names=var_vars)

forecast_steps = 8  # 8 quarters ahead

# Unconditional forecast (no bridge)
cf = ConditionalForecast(var_model, method="analytic")
unc_forecast = cf.forecast(steps=forecast_steps, conditions=None, n_draws=1000, seed=42)

# Bridged forecast: condition Q1 GDP on the nowcast
nowcast_value = float(nowcast.point[0])
bridged_forecast = cf.forecast(
    steps=forecast_steps,
    conditions={"gdp_growth": [nowcast_value]},  # condition only Q1
    n_draws=1000,
    seed=42,
)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
horizons = np.arange(1, forecast_steps + 1)

# GDP comparison
ax = axes[0]
ax.plot(horizons, unc_forecast["gdp_growth"].point, "b-o", label="Unconditional", linewidth=2)
ax.plot(horizons, bridged_forecast["gdp_growth"].point, "r-s", label=f"Bridged (nowcast={nowcast_value:.2f})", linewidth=2)
if bridged_forecast["gdp_growth"].lower_80 is not None:
    ax.fill_between(horizons, bridged_forecast["gdp_growth"].lower_80,
                    bridged_forecast["gdp_growth"].upper_80, alpha=0.15, color="red")
ax.axhline(nowcast_value, color="red", linestyle="--", alpha=0.5, label="Nowcast")
ax.set_title("GDP Growth: Unconditional vs Bridged", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# Inflation comparison
ax = axes[1]
ax.plot(horizons, unc_forecast["inflation"].point, "b-o", label="Unconditional", linewidth=2)
ax.plot(horizons, bridged_forecast["inflation"].point, "r-s", label="Bridged", linewidth=2)
ax.set_title("Inflation: Impact of GDP Bridge", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("Inflation (%)")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("Nowcast Bridge: Conditioning VAR on DFM Nowcast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"GDP Q1 — Unconditional: {unc_forecast['gdp_growth'].point[0]:.4f}")
print(f"GDP Q1 — Bridged (nowcast): {bridged_forecast['gdp_growth'].point[0]:.4f}")

## 5. Step 4: Scenario Analysis

Starting from the nowcast, we build **three scenarios** for the future:

- **Baseline**: unconditional VAR forecast from the nowcast
- **Recession**: GDP contracts, unemployment rises, rates cut aggressively
- **Boom**: GDP accelerates, low unemployment, rates rise moderately

Each scenario conditions specific variables over the forecast horizon.

In [ ]:
# Build scenarios conditional on the nowcast
builder = ScenarioBuilder(var_model)

# Baseline: unconditional from nowcast bridge
baseline_gdp = bridged_forecast["gdp_growth"].point.tolist()
builder.add_scenario("baseline",
                     {"gdp_growth": [nowcast_value]},
                     description="Unconditional from nowcast")

# Recession: GDP negative, rising unemployment
recession_gdp = [nowcast_value - 0.5 * (t + 1) for t in range(forecast_steps)]
last_unemp = us_macro["unemployment"].iloc[-1]
recession_unemp = [last_unemp + 0.3 * (t + 1) for t in range(forecast_steps)]
builder.add_scenario("recession",
                     {"gdp_growth": recession_gdp, "unemployment": recession_unemp},
                     description="Contraction with rising unemployment")

# Boom: strong growth, low unemployment
boom_gdp = [nowcast_value + 0.4 * (t + 1) for t in range(forecast_steps)]
last_ff = us_macro["fed_funds"].iloc[-1]
boom_rates = [last_ff + 0.25 * (t + 1) for t in range(forecast_steps)]
builder.add_scenario("boom",
                     {"gdp_growth": boom_gdp, "fed_funds": boom_rates},
                     description="Strong expansion with rate hikes")

# Run all scenarios
scenario_results = builder.run(steps=forecast_steps, n_draws=1000, seed=42)

# Plot with fan charts
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
colors = {"baseline": "blue", "recession": "red", "boom": "green"}
horizons = np.arange(1, forecast_steps + 1)

for idx, var in enumerate(var_vars):
    ax = axes[idx // 2, idx % 2]
    for scen_name in ["baseline", "recession", "boom"]:
        fc = scenario_results.get(scen_name, var)
        ax.plot(horizons, fc.point, "-o", color=colors[scen_name],
                label=scen_name.capitalize(), linewidth=2, markersize=4)
        if fc.lower_80 is not None:
            ax.fill_between(horizons, fc.lower_80, fc.upper_80,
                            alpha=0.1, color=colors[scen_name])
    ax.set_title(var.replace("_", " ").title(), fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Scenario Analysis: Baseline vs Recession vs Boom (conditional on nowcast)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Step 5: Probability Assessment

Using **Monte Carlo simulation**, we compute the probability of adverse events
conditional on the current nowcast.

Key question: **What is P(recession) given our nowcast?**

We simulate 5,000 stochastic paths from the VAR and count the fraction that satisfy
each event definition at each horizon.

In [ ]:
# Monte Carlo simulation
mc = MonteCarlo(var_model, n_paths=5000, seed=42, parametric=True)
paths = mc.simulate(steps=forecast_steps)
print(f"Monte Carlo paths: {paths.shape}")
print(f"  -> {paths.shape[0]} paths x {paths.shape[1]} steps x {paths.shape[2]} variables")

# Event probabilities
prob_recession = mc.probability(lambda y: y < 0.0, variable="gdp_growth")
prob_high_inf = mc.probability(lambda y: y > 4.0, variable="inflation")
prob_high_unemp = mc.probability(lambda y: y > 8.0, variable="unemployment")

# Joint probability: recession AND high inflation (stagflation)
gdp_idx = var_vars.index("gdp_growth")
inf_idx = var_vars.index("inflation")
joint_mask = (paths[:, :, gdp_idx] < 0.0) & (paths[:, :, inf_idx] > 4.0)
prob_stagflation = joint_mask.mean(axis=0)

# Results table
prob_df = pd.DataFrame({
    "P(GDP < 0%)": prob_recession,
    "P(Inflation > 4%)": prob_high_inf,
    "P(Unemployment > 8%)": prob_high_unemp,
    "P(Stagflation)": prob_stagflation,
}, index=[f"Q+{h+1}" for h in range(forecast_steps)])

print("=== Conditional Event Probabilities (given nowcast) ===")
print(prob_df.round(4).to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Event probabilities by horizon
ax = axes[0]
ax.plot(horizons, prob_recession, "r-o", label="P(Recession)", linewidth=2)
ax.plot(horizons, prob_high_inf, "orange", marker="s", label="P(High Inflation)", linewidth=2)
ax.plot(horizons, prob_high_unemp, "purple", marker="^", label="P(High Unemployment)", linewidth=2)
ax.plot(horizons, prob_stagflation, "k--", marker="D", label="P(Stagflation)", linewidth=2)
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("Probability")
ax.set_title("Event Probabilities by Forecast Horizon", fontsize=12, fontweight="bold")
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: GDP density at Q+4 with recession threshold
ax = axes[1]
gdp_draws_q4 = paths[:, 3, gdp_idx]
ax.hist(gdp_draws_q4, bins=50, density=True, alpha=0.5, color="steelblue", edgecolor="white")
kde = gaussian_kde(gdp_draws_q4)
x_range = np.linspace(gdp_draws_q4.min() - 1, gdp_draws_q4.max() + 1, 200)
ax.plot(x_range, kde(x_range), "navy", linewidth=2)
ax.axvline(0, color="red", linestyle="--", linewidth=2, label="Recession threshold")
ax.fill_between(x_range[x_range < 0], kde(x_range[x_range < 0]), alpha=0.3, color="red",
                label=f"P(recession at Q+4) = {prob_recession[3]:.3f}")
ax.set_title("GDP Growth Density at Q+4", fontsize=12, fontweight="bold")
ax.set_xlabel("GDP Growth (%)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle("Probability Assessment via Monte Carlo (5,000 paths)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Step 6: Monitoring and Update

When a new data release arrives, the nowcast should be **updated** in real time.
We simulate the arrival of new industrial production data and show how the
nowcast, news decomposition, and scenario probabilities change.

This demonstrates the **monitoring loop** that central banks run continuously.

In [ ]:
# Simulate arrival of new data: industrial production for the latest month
# Create "before" and "after" datasets
data_before = mixed_freq.copy()
data_before.iloc[-2:, data_before.columns.get_loc("industrial_production")] = np.nan
data_before.iloc[-3:, data_before.columns.get_loc("retail_sales")] = np.nan

data_after = mixed_freq.copy()
data_after.iloc[-1:, data_after.columns.get_loc("industrial_production")] = np.nan
data_after.iloc[-2:, data_after.columns.get_loc("retail_sales")] = np.nan

# Nowcast BEFORE new data
dfm_before = DFMNowcaster(
    n_factors=1, factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum", em_iterations=100,
)
dfm_before.fit(data_before)
nowcast_before = dfm_before.nowcast(target="gdp_growth")

# Nowcast AFTER new data
dfm_after = DFMNowcaster(
    n_factors=1, factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum", em_iterations=100,
)
dfm_after.fit(data_after)
nowcast_after = dfm_after.nowcast(target="gdp_growth")

# News decomposition for the update
news_update = NewsDecomposition(dfm_after)
update_result = news_update.decompose(data_before, data_after, target="gdp_growth")

print("=== Nowcast Update ===")
print(f"Before new data: {nowcast_before.point[0]:.4f}  "
      f"(95% CI: [{nowcast_before.lower_95[0]:.4f}, {nowcast_before.upper_95[0]:.4f}])")
print(f"After new data:  {nowcast_after.point[0]:.4f}  "
      f"(95% CI: [{nowcast_after.lower_95[0]:.4f}, {nowcast_after.upper_95[0]:.4f}])")
print(f"Revision:        {update_result.total_revision:+.4f}")

# Update scenario probabilities with new nowcast
new_nowcast_value = float(nowcast_after.point[0])
mc_updated = MonteCarlo(var_model, n_paths=5000, seed=42, parametric=True)
paths_updated = mc_updated.simulate(steps=forecast_steps)
prob_recession_updated = mc_updated.probability(lambda y: y < 0.0, variable="gdp_growth")

# Compare before/after
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Nowcast comparison
ax = axes[0]
labels = ["Before", "After"]
points = [nowcast_before.point[0], nowcast_after.point[0]]
ci_low = [nowcast_before.lower_95[0], nowcast_after.lower_95[0]]
ci_high = [nowcast_before.upper_95[0], nowcast_after.upper_95[0]]
errors = [[p - l for p, l in zip(points, ci_low)],
          [h - p for p, h in zip(points, ci_high)]]
ax.bar(labels, points, color=["steelblue", "darkorange"], edgecolor="white", width=0.5)
ax.errorbar(labels, points, yerr=errors, fmt="none", color="black", capsize=8)
ax.set_title("Nowcast: Before vs After Update", fontsize=12, fontweight="bold")
ax.set_ylabel("GDP Growth (%)")
for i, v in enumerate(points):
    ax.text(i, v + 0.05, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")

# Panel 2: News contributions
update_result.plot_contributions(ax=axes[1])
axes[1].set_title("News Contributions", fontsize=12, fontweight="bold")

# Panel 3: Recession probability comparison
ax = axes[2]
ax.plot(horizons, prob_recession, "b-o", label="Before update", linewidth=2)
ax.plot(horizons, prob_recession_updated, "r-s", label="After update", linewidth=2)
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("P(Recession)")
ax.set_title("Recession Probability: Before vs After", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.02, 1.02)

fig.suptitle("Real-Time Monitoring: Impact of New Data Release",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n=== Summary ===")
print(f"New data release shifted the GDP nowcast by {update_result.total_revision:+.4f}.")
print(f"P(recession at Q+4): {prob_recession[3]:.3f} -> {prob_recession_updated[3]:.3f}")

## Exercise 1: Add MIDAS to the nowcasting step

Replace the DFM nowcaster with a **MIDAS (Mixed Data Sampling)** model from
`forecastbox.nowcasting`. Compare the MIDAS and DFM nowcasts, then bridge
both to the VAR for scenario analysis. Which nowcaster produces better results?

In [ ]:
# TODO: Exercise 1
# Steps:
# 1. from forecastbox.nowcasting import MIDAS
# 2. Fit MIDAS on mixed_freq.csv with gdp_growth as target
# 3. Generate MIDAS nowcast and compare to DFM nowcast
# 4. Bridge both nowcasts to the VAR model
# 5. Compare scenario probabilities under each nowcaster

## Exercise 2: Build stress test conditional on nowcast

Using `forecastbox.scenarios.StressTest`, build a stress scenario where:
- GDP drops by 3 standard deviations below the nowcast
- Unemployment rises by 2 standard deviations

Compute the impulse response functions and expected shortfall for inflation
under this stress scenario.

In [ ]:
# TODO: Exercise 2
# Steps:
# 1. from forecastbox.scenarios import StressTest, Shock
# 2. Define shocks: GDP -3 sigma, unemployment +2 sigma
# 3. StressTest(var_model).run(shocks=..., steps=8)
# 4. Plot IRFs for all variables
# 5. Compute expected shortfall for inflation using MonteCarlo